In [ ]:
#!/usr/bin/env python3
"""
Spatial Transcriptomics Analysis Pipeline — Robust AD Gene Discovery
=====================================================================
Mouse Brain - Alzheimer's Disease Study (Publication-Ready)

4 groups × 4 replicates = 16 samples:
  - Young Control (YC)
  - Young AD (YAD)
  - Aged Control (AC)
  - Aged AD (AAD)

Key methodological choices:
  - Pseudobulk DE via pyDESeq2 (negative binomial, proper shrinkage)
  - Factorial design: ~Age + Condition (+ optional interaction)
  - Per-sample Moran's I with Fisher's method aggregation
  - Direction concordance filtering for robust gene identification
  - Batch-corrected embedding via Harmony

Goal:
Identify robust Alzheimer's Disease genes that:
  1. Are DE via pyDESeq2 factorial model (Condition effect)
  2. Optionally validated by per-age-group contrasts with direction concordance
  3. Survive FDR correction (padj < 0.05)
  4. Have |log2FC| > 0.5
  5. Are spatially variable in AD tissue (per-sample Moran's I)
"""

# =============================================================================
# 0. Imports and Setup
# =============================================================================

import scanpy as sc
import squidpy as sq
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import sparse
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import harmonypy as hm
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)

FIGURE_DIR = "figures"
RESULTS_DIR = "results"
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
sc.settings.figdir = FIGURE_DIR

# =============================================================================
# 1. Load and Concatenate Samples
# =============================================================================

input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"

sample_files = {
    "YC_1": "A1_Young_Control_Mouse_Brain_202502.h5ad",
    "YC_2": "B1_Young_Control_Mouse_Brain_202502.h5ad",
    "YC_3": "C1_Young_Control_Mouse_Brain_202502.h5ad",
    "YC_4": "D1_Young_Control_Mouse_Brain_202502.h5ad",
    "YAD_1": "A1_Young_AD_Mouse_Brain_202502.h5ad",
    "YAD_2": "B1_Young_AD_Mouse_Brain_202502.h5ad",
    "YAD_3": "C1_Young_AD_Mouse_Brain_202502.h5ad",
    "YAD_4": "D1_Young_AD_Mouse_Brain_202502.h5ad",
    "AC_1": "A1_Aged_Control_Mouse_Brain_202502.h5ad",
    "AC_2": "B1_Aged_Control_Mouse_Brain_202502.h5ad",
    "AC_3": "C1_Aged_Control_Mouse_Brain_202502.h5ad",
    "AC_4": "D1_Aged_Control_Mouse_Brain_202502.h5ad",
    "AAD_1": "A1_Aged_AD_Mouse_Brain_202502.h5ad",
    "AAD_2": "B1_Aged_AD_Mouse_Brain_202502.h5ad",
    "AAD_3": "C1_Aged_AD_Mouse_Brain_202502.h5ad",
    "AAD_4": "D1_Aged_AD_Mouse_Brain_202502.h5ad",
}

sample_metadata = {
    s: {
        "Group": s.split("_")[0],
        "Condition": "AD" if "AD" in s else "Control",
        "Age": "Young" if s.startswith("Y") else "Aged",
    }
    for s in sample_files
}

print("Loading samples...")
adatas = []

for sample_name, filename in sample_files.items():
    filepath = os.path.join(input_folder, filename)
    a = sc.read_h5ad(filepath)
    a.obs["Sample"] = sample_name

    for key, val in sample_metadata[sample_name].items():
        a.obs[key] = val

    a.obs_names = [f"{sample_name}_{x}" for x in a.obs_names]
    a.obs_names_make_unique()
    adatas.append(a)

adata = ad.concat(adatas, join="inner", merge="first")
adata.obs_names_make_unique()
print(f"Concatenated: {adata.shape}")

# =============================================================================
# 2. Quality Control
# =============================================================================

print("\n=== Quality Control ===")

# Remove zero-count genes
if sparse.issparse(adata.X):
    gene_mask = np.array(adata.X.sum(axis=0) != 0).ravel()
else:
    gene_mask = adata.X.sum(axis=0) != 0

adata = adata[:, gene_mask].copy()

# Annotate MT and ribosomal genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
adata.var["ribo"] = adata.var_names.str.match("^(Rpl|Rps)")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

# Record pre-filter shape
pre_filter = adata.shape
print(f"  Pre-filter: {pre_filter}")

# Apply QC filters
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()

print(f"  Post-filter: {adata.shape}")
print(f"  Removed {pre_filter[0] - adata.shape[0]} cells, "
      f"{pre_filter[1] - adata.shape[1]} genes")

# Optional: QC violin plots per sample
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    sc.pl.violin(adata, metric, groupby="Sample", rotation=90, ax=ax, show=False)
    ax.set_title(metric)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "QC_violins.png"), dpi=200, bbox_inches="tight")
plt.close()

# =============================================================================
# 3. Doublet Detection (Scrublet)
# =============================================================================

print("\n=== Doublet Detection ===")

import scrublet as scr

doublet_scores = []
predicted_doublets = []

for sample in adata.obs["Sample"].unique():
    mask = adata.obs["Sample"] == sample
    sub = adata[mask].copy()

    if sparse.issparse(sub.X):
        counts_matrix = sub.X.toarray()
    else:
        counts_matrix = sub.X

    scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.06)
    scores, preds = scrub.scrub_doublets(min_counts=2, min_cells=3, verbose=False)

    doublet_scores.extend(scores)
    predicted_doublets.extend(preds)

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets

n_doublets = sum(predicted_doublets)
print(f"  Detected {n_doublets} doublets ({100*n_doublets/adata.shape[0]:.1f}%)")

# Remove doublets
adata = adata[~adata.obs["predicted_doublet"]].copy()
print(f"  Post-doublet removal: {adata.shape}")

# =============================================================================
# 4. Normalization
# =============================================================================

print("\n=== Normalization ===")

# Store raw counts BEFORE any normalization
adata.layers["raw_counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.layers["log_norm"] = adata.X.copy()
adata.raw = adata.copy()

# =============================================================================
# 5. Feature Selection
# =============================================================================

print("\n=== Feature Selection ===")

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    batch_key="Sample",
    layer="raw_counts",
)

n_hvg = adata.var["highly_variable"].sum()
print(f"  HVGs selected: {n_hvg}")

# =============================================================================
# 6. PCA + Harmony Batch Correction + Clustering
# =============================================================================

print("\n=== Dimensionality Reduction & Clustering ===")

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50)

# --- Harmony batch correction ---
print("  Running Harmony integration...")
harmony_out = hm.run_harmony(
    adata.obsm["X_pca"],
    adata.obs,
    "Sample",
    max_iter_harmony=30,
)
adata.obsm["X_pca_harmony"] = harmony_out.Z_corr

# Use Harmony-corrected PCA for neighbors/UMAP
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

print(f"  Clusters found: {adata.obs['leiden'].nunique()}")

# UMAP plots
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
sc.pl.umap(adata, color="leiden", ax=axes[0], show=False, title="Clusters")
sc.pl.umap(adata, color="Group", ax=axes[1], show=False, title="Group")
sc.pl.umap(adata, color="Sample", ax=axes[2], show=False, title="Sample")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "UMAP_overview.png"), dpi=200, bbox_inches="tight")
plt.close()

# =============================================================================
# 7. Build Pseudobulk (Raw Counts)
# =============================================================================

print("\n=== Building Pseudobulk ===")


def build_pseudobulk_raw(adata):
    """Build pseudobulk from raw counts, returning integer count matrix."""
    samples = adata.obs["Sample"].unique()
    pb_data = []
    pb_obs = []

    for sample in samples:
        mask = adata.obs["Sample"] == sample
        n_cells = mask.sum()

        if sparse.issparse(adata.layers["raw_counts"]):
            counts = np.array(
                adata.layers["raw_counts"][mask].sum(axis=0)
            ).ravel()
        else:
            counts = adata.layers["raw_counts"][mask].sum(axis=0).ravel()

        pb_data.append(counts)

        row = adata.obs.loc[mask].iloc[0]
        pb_obs.append({
            "Group": row["Group"],
            "Condition": row["Condition"],
            "Age": row["Age"],
            "n_cells": n_cells,
        })

    pb = ad.AnnData(
        X=np.array(pb_data, dtype=np.float64),
        obs=pd.DataFrame(pb_obs, index=samples),
        var=adata.var[[]].copy(),  # just gene names, no extra columns
    )

    return pb


pb = build_pseudobulk_raw(adata)
print(f"  Pseudobulk shape: {pb.shape}")
print(f"  Cells per sample:\n{pb.obs['n_cells'].to_string()}")

# Filter lowly expressed genes in pseudobulk
# Keep genes with at least 10 counts in at least 4 samples (one full group)
gene_detected = (pb.X >= 10).sum(axis=0)
keep_genes = gene_detected >= 4
pb = pb[:, keep_genes].copy()
print(f"  Pseudobulk after gene filter: {pb.shape}")

# =============================================================================
# 8. Differential Expression with pyDESeq2
# =============================================================================

print("\n=== Differential Expression (pyDESeq2) ===")

# --- 8a. Factorial Model: ~Age + Condition ---
print("\n--- Factorial Model: ~Age + Condition ---")

pb_factorial = pb.copy()

# Ensure factors are categorical
pb_factorial.obs["Age"] = pd.Categorical(
    pb_factorial.obs["Age"], categories=["Young", "Aged"]
)
pb_factorial.obs["Condition"] = pd.Categorical(
    pb_factorial.obs["Condition"], categories=["Control", "AD"]
)

# Round counts to integers (required by pyDESeq2)
pb_factorial.X = np.round(pb_factorial.X).astype(int)

dds_factorial = DeseqDataSet(
    adata=pb_factorial,
    design="~Age + Condition",
    refit_cooks=True,
    quiet=True,
)
dds_factorial.deseq2()

# Extract Condition effect (AD vs Control, accounting for Age)
stat_condition = DeseqStats(
    dds_factorial,
    contrast=["Condition", "AD", "Control"],
    quiet=True,
)
stat_condition.summary()
factorial_results = stat_condition.results_df.copy()
factorial_results["gene"] = factorial_results.index

print(f"  Total genes tested: {factorial_results.shape[0]}")

factorial_sig = factorial_results[
    (factorial_results["padj"] < 0.05)
    & (np.abs(factorial_results["log2FoldChange"]) > 0.5)
].copy()
print(f"  Significant (padj<0.05, |log2FC|>0.5): {factorial_sig.shape[0]}")

factorial_results.to_csv(
    os.path.join(RESULTS_DIR, "DESeq2_factorial_all.csv")
)
factorial_sig.to_csv(
    os.path.join(RESULTS_DIR, "DESeq2_factorial_significant.csv")
)

# --- 8b. Per-Age Contrasts for Validation ---
print("\n--- Per-Age Contrasts (Validation) ---")


def run_pydeseq2_contrast(pb, subset_col, subset_val, contrast_col, ref, test):
    """Run pyDESeq2 on a subset of pseudobulk samples."""
    mask = pb.obs[subset_col] == subset_val
    pb_sub = pb[mask].copy()

    pb_sub.obs[contrast_col] = pd.Categorical(
        pb_sub.obs[contrast_col], categories=[ref, test]
    )
    pb_sub.X = np.round(pb_sub.X).astype(int)

    # Filter genes with very low counts in this subset
    gene_ok = (pb_sub.X >= 5).sum(axis=0) >= 2
    pb_sub = pb_sub[:, gene_ok].copy()

    dds = DeseqDataSet(
        adata=pb_sub,
        design=f"~{contrast_col}",
        refit_cooks=True,
        quiet=True,
    )
    dds.deseq2()

    stat = DeseqStats(
        dds,
        contrast=[contrast_col, test, ref],
        quiet=True,
    )
    stat.summary()
    res = stat.results_df.copy()
    res["gene"] = res.index
    return res


# Young: AD vs Control
young_results = run_pydeseq2_contrast(
    pb, "Age", "Young", "Condition", "Control", "AD"
)
young_sig = young_results[
    (young_results["padj"] < 0.05)
    & (np.abs(young_results["log2FoldChange"]) > 0.5)
].copy()

# Aged: AD vs Control
aged_results = run_pydeseq2_contrast(
    pb, "Age", "Aged", "Condition", "Control", "AD"
)
aged_sig = aged_results[
    (aged_results["padj"] < 0.05)
    & (np.abs(aged_results["log2FoldChange"]) > 0.5)
].copy()

print(f"  Young AD vs Control significant: {young_sig.shape[0]}")
print(f"  Aged AD vs Control significant: {aged_sig.shape[0]}")

young_results.to_csv(os.path.join(RESULTS_DIR, "DESeq2_young_ADvsControl.csv"))
aged_results.to_csv(os.path.join(RESULTS_DIR, "DESeq2_aged_ADvsControl.csv"))

# --- 8c. Direction Concordance Check ---
print("\n--- Direction Concordance ---")

# Genes significant in BOTH age groups
common_genes = set(young_sig["gene"]).intersection(set(aged_sig["gene"]))
print(f"  Genes significant in both contrasts: {len(common_genes)}")

if len(common_genes) > 0:
    merged_contrasts = young_sig[young_sig["gene"].isin(common_genes)].merge(
        aged_sig[aged_sig["gene"].isin(common_genes)],
        on="gene",
        suffixes=("_young", "_aged"),
    )

    # Keep only concordant direction
    concordant = merged_contrasts[
        np.sign(merged_contrasts["log2FoldChange_young"])
        == np.sign(merged_contrasts["log2FoldChange_aged"])
    ].copy()

    discordant = merged_contrasts[
        np.sign(merged_contrasts["log2FoldChange_young"])
        != np.sign(merged_contrasts["log2FoldChange_aged"])
    ]

    print(f"  Concordant direction: {concordant.shape[0]}")
    print(f"  Discordant direction (removed): {discordant.shape[0]}")

    concordant.to_csv(
        os.path.join(RESULTS_DIR, "concordant_per_age_genes.csv"), index=False
    )
else:
    concordant = pd.DataFrame(columns=["gene"])

# =============================================================================
# 9. Combine Evidence: Factorial + Concordance
# =============================================================================

print("\n=== Combining Evidence ===")

# Primary gene set: factorial model significant genes
primary_genes = set(factorial_sig["gene"])

# Validation gene set: concordant across age groups
validation_genes = set(concordant["gene"]) if concordant.shape[0] > 0 else set()

# Robust genes: significant in factorial AND concordant in per-age contrasts
# OR: significant in factorial (the factorial result is the primary evidence)
robust_genes_strict = primary_genes.intersection(validation_genes)
robust_genes_factorial = primary_genes  # broader set

print(f"  Factorial significant: {len(primary_genes)}")
print(f"  Per-age concordant: {len(validation_genes)}")
print(f"  Strict (factorial ∩ concordant): {len(robust_genes_strict)}")

# Use factorial as the main result; flag concordance as additional support
factorial_sig["concordant_per_age"] = factorial_sig["gene"].isin(validation_genes)

# =============================================================================
# 10. Spatial Validation — Per-Sample Moran's I
# =============================================================================

print("\n=== Spatial Validation (Per-Sample Moran's I) ===")

adata_ad = adata[adata.obs["Condition"] == "AD"].copy()

# Restore log-normalized data for Moran's I
adata_ad.X = adata_ad.layers["log_norm"].copy()

# Subset to genes of interest to speed up computation
genes_to_test = list(primary_genes.intersection(set(adata_ad.var_names)))
print(f"  Testing {len(genes_to_test)} genes for spatial autocorrelation")

moran_results = []

for sample in sorted(adata_ad.obs["Sample"].unique()):
    print(f"    Processing {sample}...")
    sub = adata_ad[adata_ad.obs["Sample"] == sample].copy()

    # Ensure spatial coordinates exist
    if "spatial" not in sub.obsm:
        print(f"    WARNING: No spatial coordinates for {sample}, skipping.")
        continue

    # Build spatial graph for this sample
    sq.gr.spatial_neighbors(sub, coord_type="generic", delaunay=True)

    # Subset to genes of interest
    genes_present = [g for g in genes_to_test if g in sub.var_names]
    if len(genes_present) == 0:
        continue

    sub_genes = sub[:, genes_present].copy()

    try:
        sq.gr.spatial_autocorr(sub_genes, mode="moran", n_perms=999, n_jobs=4)
        df = sub_genes.uns["moranI"].copy()
        df["sample"] = sample
        df["gene"] = df.index
        moran_results.append(df)
    except Exception as e:
        print(f"    WARNING: Moran's I failed for {sample}: {e}")
        continue

if len(moran_results) > 0:
    moran_all = pd.concat(moran_results, ignore_index=True)

    # Aggregate per gene across AD samples
    def aggregate_moran(gene_df):
        pvals = gene_df["pval_norm"].dropna().values
        pvals = pvals[pvals > 0]  # remove exact zeros for Fisher's method

        if len(pvals) < 2:
            combined_p = pvals[0] if len(pvals) == 1 else 1.0
        else:
            _, combined_p = combine_pvalues(pvals, method="fisher")

        return pd.Series({
            "mean_moranI": gene_df["I"].mean(),
            "median_moranI": gene_df["I"].median(),
            "combined_pval": combined_p,
            "n_samples_tested": len(gene_df),
            "n_samples_sig": (gene_df["pval_norm"] < 0.05).sum(),
        })

    moran_agg = moran_all.groupby("gene").apply(aggregate_moran).reset_index()

    # FDR correction on combined p-values
    moran_agg["padj"] = multipletests(
        moran_agg["combined_pval"], method="fdr_bh"
    )[1]

    # Require: FDR significant AND significant in at least 2 individual samples
    min_samples_sig = 2
    svg_sig = moran_agg[
        (moran_agg["padj"] < 0.05) & (moran_agg["n_samples_sig"] >= min_samples_sig)
    ].copy()

    print(f"  Spatially variable genes (padj<0.05, ≥{min_samples_sig} samples): "
          f"{svg_sig.shape[0]}")

    moran_agg.to_csv(os.path.join(RESULTS_DIR, "MoranI_aggregated.csv"), index=False)
    svg_sig.to_csv(os.path.join(RESULTS_DIR, "MoranI_significant.csv"), index=False)

else:
    print("  WARNING: No Moran's I results obtained.")
    svg_sig = pd.DataFrame(columns=["gene"])

# =============================================================================
# 11. Final Robust AD Gene List
# =============================================================================

print("\n=== Final Robust AD Gene List ===")

# Merge factorial DE results with spatial validation
spatially_validated = set(svg_sig["gene"])

final_genes = factorial_sig[
    factorial_sig["gene"].isin(spatially_validated)
].copy()

# Add spatial and concordance annotations
final_genes = final_genes.merge(
    moran_agg[["gene", "mean_moranI", "combined_pval", "n_samples_sig"]],
    on="gene",
    how="left",
    suffixes=("", "_spatial"),
)

# Composite score: combine effect size, significance, and spatial signal
final_genes["abs_log2FC"] = np.abs(final_genes["log2FoldChange"])
final_genes["neg_log10_padj"] = -np.log10(final_genes["padj"].clip(lower=1e-300))
final_genes["neg_log10_spatial_p"] = -np.log10(
    final_genes["combined_pval"].clip(lower=1e-300)
)

# Rank by composite: normalize each component to [0, 1] and average
for col in ["abs_log2FC", "neg_log10_padj", "neg_log10_spatial_p", "mean_moranI"]:
    cmin = final_genes[col].min()
    cmax = final_genes[col].max()
    if cmax > cmin:
        final_genes[f"{col}_norm"] = (final_genes[col] - cmin) / (cmax - cmin)
    else:
        final_genes[f"{col}_norm"] = 0.5

final_genes["composite_score"] = (
    final_genes["abs_log2FC_norm"] * 0.3
    + final_genes["neg_log10_padj_norm"] * 0.3
    + final_genes["neg_log10_spatial_p_norm"] * 0.2
    + final_genes["mean_moranI_norm"] * 0.2
)

final_genes = final_genes.sort_values("composite_score", ascending=False)

print(f"  Final robust + spatial AD genes: {final_genes.shape[0]}")

# Save full list
final_genes.to_csv(
    os.path.join(RESULTS_DIR, "Final_Robust_Spatial_AD_genes.csv"), index=False
)

# =============================================================================
# 12. Select Top 20 and Report
# =============================================================================

print("\n=== Top 20 Robust Spatial AD Genes ===")

top20 = final_genes.head(20).copy()

# Clean output columns
top20_report = top20[[
    "gene", "log2FoldChange", "padj", "concordant_per_age",
    "mean_moranI", "n_samples_sig", "composite_score"
]].copy()

top20_report.columns = [
    "Gene", "log2FC", "FDR", "Concordant_Across_Ages",
    "Mean_Moran_I", "N_Spatial_Samples", "Composite_Score"
]

top20_report.to_csv(
    os.path.join(RESULTS_DIR, "Top20_Robust_AD_genes.csv"), index=False
)

print(top20_report.to_string(index=False))

# =============================================================================
# 13. Publication Figures
# =============================================================================

print("\n=== Generating Figures ===")

top_gene_list = top20_report["Gene"].tolist()

# Restore log-normalized data for plotting
adata.X = adata.layers["log_norm"].copy()

# --- 13a. Dotplot ---
if len(top_gene_list) > 0:
    sc.pl.dotplot(
        adata,
        var_names=top_gene_list,
        groupby="Group",
        standard_scale="var",
        use_raw=True,
        dendrogram=True,
        show=False,
    )
    plt.savefig(
        os.path.join(FIGURE_DIR, "Top20_Dotplot.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()
    print("  Saved: Top20_Dotplot.png")

# --- 13b. Volcano Plot (Factorial) ---
fig, ax = plt.subplots(figsize=(10, 8))

factorial_plot = factorial_results.dropna(subset=["padj", "log2FoldChange"]).copy()
factorial_plot["neg_log10_padj"] = -np.log10(factorial_plot["padj"].clip(lower=1e-300))

# Color categories
colors = []
for _, row in factorial_plot.iterrows():
    if row["padj"] < 0.05 and abs(row["log2FoldChange"]) > 0.5:
        if row["gene"] in set(top20_report["Gene"]):
            colors.append("red")
        elif row["log2FoldChange"] > 0:
            colors.append("salmon")
        else:
            colors.append("steelblue")
    else:
        colors.append("grey")

ax.scatter(
    factorial_plot["log2FoldChange"],
    factorial_plot["neg_log10_padj"],
    c=colors,
    s=8,
    alpha=0.6,
    edgecolors="none",
)

# Label top genes
for _, row in top20_report.head(10).iterrows():
    gene = row["Gene"]
    if gene in factorial_plot["gene"].values:
        gdata = factorial_plot[factorial_plot["gene"] == gene].iloc[0]
        ax.annotate(
            gene,
            (gdata["log2FoldChange"], gdata["neg_log10_padj"]),
            fontsize=7,
            fontweight="bold",
            ha="center",
            va="bottom",
        )

ax.axhline(-np.log10(0.05), ls="--", color="grey", lw=0.8)
ax.axvline(-0.5, ls="--", color="grey", lw=0.8)
ax.axvline(0.5, ls="--", color="grey", lw=0.8)
ax.set_xlabel("log2 Fold Change (AD vs Control)")
ax.set_ylabel("-log10(FDR)")
ax.set_title("Factorial DE: AD vs Control (accounting for Age)")
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "Volcano_Factorial.png"), dpi=300, bbox_inches="tight"
)
plt.close()
print("  Saved: Volcano_Factorial.png")

# --- 13c. Heatmap of Top 20 ---
if len(top_gene_list) > 0:
    sc.pl.heatmap(
        adata,
        var_names=top_gene_list,
        groupby="Group",
        standard_scale="var",
        use_raw=True,
        show=False,
        figsize=(12, 6),
    )
    plt.savefig(
        os.path.join(FIGURE_DIR, "Top20_Heatmap.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()
    print("  Saved: Top20_Heatmap.png")

# --- 13d. Spatial Plots for Top Genes (per sample) ---
print("  Generating spatial expression plots...")

# Plot top 5 genes across AD samples
top5_genes = top_gene_list[:5]
ad_samples = sorted(adata_ad.obs["Sample"].unique())

for gene in top5_genes:
    if gene not in adata_ad.var_names:
        continue

    n_samples = len(ad_samples)
    fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 4.5))
    if n_samples == 1:
        axes = [axes]

    for ax, sample in zip(axes, ad_samples):
        sub = adata_ad[adata_ad.obs["Sample"] == sample].copy()
        if "spatial" in sub.obsm:
            sc.pl.spatial(
                sub,
                color=gene,
                spot_size=1.0,
                ax=ax,
                show=False,
                title=f"{gene} - {sample}",
                use_raw=True,
            )
        else:
            ax.set_title(f"{gene} - {sample}\n(no spatial coords)")
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(
        os.path.join(FIGURE_DIR, f"Spatial_{gene}.png"),
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

print("  Saved: Spatial expression plots for top 5 genes")

# --- 13e. Effect Size Comparison Plot ---
if concordant.shape[0] > 0 and len(top_gene_list) > 0:
    concordant_top = concordant[concordant["gene"].isin(top_gene_list)].copy()

    if concordant_top.shape[0] > 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(
            concordant_top["log2FoldChange_young"],
            concordant_top["log2FoldChange_aged"],
            s=50,
            c="darkred",
            alpha=0.7,
        )

        for _, row in concordant_top.iterrows():
            ax.annotate(
                row["gene"],
                (row["log2FoldChange_young"], row["log2FoldChange_aged"]),
                fontsize=7,
                ha="center",
                va="bottom",
            )

        lims = [
            min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1]),
        ]
        ax.plot(lims, lims, "k--", alpha=0.3, lw=1)
        ax.axhline(0, color="grey", lw=0.5)
        ax.axvline(0, color="grey", lw=0.5)
        ax.set_xlabel("log2FC (Young AD vs Young Control)")
        ax.set_ylabel("log2FC (Aged AD vs Aged Control)")
        ax.set_title("Effect Size Concordance — Top AD Genes")
        plt.tight_layout()
        plt.savefig(
            os.path.join(FIGURE_DIR, "EffectSize_Concordance.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()
        print("  Saved: EffectSize_Concordance.png")

# =============================================================================
# 14. Summary Statistics
# =============================================================================

print("\n" + "=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Total cells after QC + doublet removal: {adata.shape[0]:,}")
print(f"Total genes after filtering: {adata.shape[1]:,}")
print(f"Pseudobulk samples: {pb.shape[0]}")
print(f"Factorial DE significant: {factorial_sig.shape[0]}")
print(f"Per-age concordant: {concordant.shape[0]}")
print(f"Spatially variable (aggregated Moran's I): {svg_sig.shape[0]}")
print(f"Final robust + spatial AD genes: {final_genes.shape[0]}")
print(f"Top 20 reported: {min(20, final_genes.shape[0])}")
print("=" * 60)

# Save session info
session_info = {
    "total_cells": adata.shape[0],
    "total_genes": adata.shape[1],
    "n_samples": pb.shape[0],
    "n_factorial_sig": factorial_sig.shape[0],
    "n_concordant": concordant.shape[0],
    "n_spatial_sig": svg_sig.shape[0],
    "n_final_genes": final_genes.shape[0],
}
pd.Series(session_info).to_csv(os.path.join(RESULTS_DIR, "pipeline_summary.csv"))

print("\nPipeline Complete — All results saved to:", RESULTS_DIR)
print("Figures saved to:", FIGURE_DIR)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject